# Document Intelligence Pipeline — Demo Notebook

This notebook walks through every stage of the pipeline interactively.  
Use it to test individual components, inspect intermediate results, and run the full pipeline.

---

## 1. Setup & Configuration

In [ ]:
import sys, logging
sys.path.insert(0, '..')  # so we can import the src package

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s  %(levelname)-8s  %(message)s',
    datefmt='%H:%M:%S'
)

from src import PipelineConfig, PDFProcessor, AIEngine, DocumentPipeline

# All settings in one place — override any defaults here
cfg = PipelineConfig(
    input_dir='../input_pdfs',
    output_dir='../output',
    glyph_ratio_threshold=0.30,
    force_ocr=False,
    max_workers=1,       # set to 4+ for production
)
print(f'Input:  {cfg.input_dir}')
print(f'Output: {cfg.output_dir}')
print(f'Model:  {cfg.model_classify}')

---
## 2. PDF Type Detection

The `PDFProcessor` analyses each page to determine which extraction method will produce the cleanest text.  
This is the key step that **minimises token waste** — sending glyph-garbage to the LLM would burn tokens for nothing.

| PDF Type | Detection Logic | Extraction Route |
|----------|----------------|------------------|
| `text_normal` | Clean text, high alphanumeric ratio | PyMuPDF (1–2 sec) |
| `text_glyph` | Text exists but ratio < 0.30 or < 5 words | Tesseract OCR |
| `scanned` | No text layer, has images | Tesseract OCR |
| `hybrid` | Both text and image layers | Tesseract OCR |
| `empty` | No content at all | Error |

In [ ]:
from pathlib import Path

proc = PDFProcessor(cfg)

# Analyse all PDFs in input directory
input_dir = Path(cfg.input_dir)
if input_dir.exists():
    for pdf in sorted(input_dir.glob('*.pdf')):
        # Split first, then detect each page
        pages = proc.split_pdf(pdf)
        for page in pages:
            analysis = proc.detect_type(page)
            print(f'{page.name:40s}  →  {analysis.pdf_type:12s}  '
                  f'(α-ratio: {analysis.alphanumeric_ratio:.2f}, '
                  f'words: {analysis.readable_words})')
    print('\nStats:', proc.get_stats())
else:
    print(f'Put PDF files in {input_dir} and re-run this cell.')

---
## 3. Text Extraction Deep Dive

Process a single page through the full extraction workflow and inspect the raw text output.

In [ ]:
# Pick the first PDF page to inspect
if input_dir.exists():
    first_pdf = next(input_dir.glob('*.pdf'), None)
    if first_pdf:
        pages = proc.split_pdf(first_pdf)
        result = proc.process_page(pages[0])
        
        print(f'File:       {result.source_file}')
        print(f'PDF Type:   {result.pdf_type}')
        print(f'Method:     {result.extraction_method}')
        print(f'Notes:      {result.notes}')
        print(f'Has text:   {result.text_content is not None}')
        print(f'Has image:  {result.base64_image is not None}')
        if result.text_content:
            print(f'\n--- Extracted Text (first 500 chars) ---')
            print(result.text_content[:500])
    else:
        print('No PDFs found.')
else:
    print(f'Put PDF files in {input_dir}')

---
## 4. AI Classification & Extraction

The `AIEngine` sends the cleaned text (or base64 image) to Claude for:
1. **Classification** — what type of document is this?
2. **Structured extraction** — pull out the schema-specific fields.

Each call is kept small by using separate prompts with tight `max_tokens`.

In [ ]:
ai = AIEngine(cfg)

# Use the text from the previous cell
if 'result' in dir() and result.text_content:
    ai_result = ai.classify_and_extract(
        content=result.text_content,
        filename=result.source_file,
        format_type='TEXT'
    )
    
    print(f"Type:       {ai_result['document_type']}")
    print(f"Confidence: {ai_result['confidence']:.0%}")
    print(f"Status:     {ai_result['status']}")
    if 'extracted_data' in ai_result:
        print(f'\n--- Extracted Fields ---')
        for k, v in ai_result['extracted_data'].items():
            if not k.startswith('_'):
                print(f'  {k:25s}: {v}')
else:
    print('No text content available. Run cell 3 first.')

---
## 5. Run the Full Pipeline

One call processes every PDF in the input directory:  
`split → detect → extract text → classify → extract fields → validate → CSV`

Results are written to CSV files in the output directory.

In [ ]:
pipeline = DocumentPipeline(cfg)
stats = pipeline.run()

print(f'\nDone! Check {cfg.output_dir}/ for CSV outputs.')

---
## 6. Inspect Output CSVs

In [ ]:
import csv
from pathlib import Path

output = Path(cfg.output_dir)
for csv_file in sorted(output.glob('*.csv')):
    with open(csv_file, encoding='utf-8') as f:
        rows = list(csv.DictReader(f))
    if rows:
        print(f'\n=== {csv_file.name} ({len(rows)} records) ===')
        for row in rows[:3]:  # show first 3
            for k, v in row.items():
                if v:
                    print(f'  {k:25s}: {v}')
            print()

---

## 7. Configuration Reference

Every parameter is set in `PipelineConfig`. Override at instantiation:

```python
cfg = PipelineConfig(
    input_dir='./my_pdfs',
    output_dir='./results',
    glyph_ratio_threshold=0.25,   # lower = more aggressive glyph detection
    force_ocr=True,               # always use Tesseract
    max_workers=8,                # concurrent PDFs
    model_classify='claude-sonnet-4-20250514',
    confidence_threshold=0.80,    # stricter classification
)
```

### Adding a New Document Type

1. Add a schema entry to `DOCUMENT_SCHEMAS` in `src/config.py`
2. Add indicators to `CLASSIFICATION_SYSTEM_PROMPT`
3. That's it — no code changes needed in the pipeline

### Token Optimisation Tips

| Strategy | Impact |
|----------|--------|
| Glyph detection routes bad PDFs to OCR | Prevents sending garbage text to Claude |
| Text extraction before vision API | Text calls are ~10x cheaper than image calls |
| Separate classify/extract with tight max_tokens | Avoids over-generation |
| Type-specific schemas | Claude only extracts relevant fields |